# Phase 4 — SQL Insights Layer

**What this notebook does:**
- Runs business-facing SQL queries directly against Gold Delta tables
- Creates 4 reusable Unity Catalog views for BI tools
- Produces 10 cross-KPI insights answering real operational questions

**Views created:**
- `vw_fleet_health_dashboard` — per drone combined health view (success rate + battery efficiency)
- `vw_route_summary` — route-level performance view (avg time, variance, distance per route)
- `vw_failure_cause_summary` — failure cause breakdown with percentage of total deliveries
- `vw_monthly_failure_trend` — month-over-month failure and success rate across 6 months

**Insights produced:**
1. Top and bottom performing drones
2. Drone model performance comparison
3. High-risk drone identification
4. Slowest and busiest routes
5. Failure cause deep dive
6. Winter vs non-winter failure rate
7. Cross-KPI fleet health ranking
8. Destination Zone Failure Hotspots
9. Fleet Workload Distribution
10. Failure Timing: Early vs Late Flight

In [0]:
CATALOG = "ddmfa_catalog"
SCHEMA  = "drone_schema"
DB      = f"{CATALOG}.{SCHEMA}"

print(f"Querying Gold tables from : {DB}")
print(f"Writing views to          : {DB}")

Querying Gold tables from : ddmfa_catalog.drone_schema
Writing views to          : ddmfa_catalog.drone_schema


## 1. Create Reusable Unity Catalog Views
Views sit on top of Gold tables and give BI tools and analysts a stable, clean interface.
Underlying Gold tables can change without breaking the views downstream.

In [0]:
# View 1: Fleet Health Dashboard
# Joins drone success rate + battery efficiency into one analyst-ready view
spark.sql(f"""
    CREATE OR REPLACE VIEW {DB}.vw_fleet_health_dashboard AS
    SELECT
        s.drone_id,
        s.model,
        s.max_range_km,
        s.total_deliveries,
        s.successful_deliveries,
        s.total_failures,
        s.success_rate_pct,
        b.avg_km_per_pct_battery,
        b.avg_battery_consumed_pct,
        b.avg_distance_km
    FROM {DB}.gold_delivery_success_rate s
    LEFT JOIN {DB}.gold_battery_efficiency b
        ON s.drone_id = b.drone_id
""")

print("vw_fleet_health_dashboard created")
spark.sql(f"SELECT * FROM {DB}.vw_fleet_health_dashboard LIMIT 3").show(truncate=False)

vw_fleet_health_dashboard created
+--------+---------+------------+----------------+---------------------+--------------+----------------+----------------------+------------------------+---------------+
|drone_id|model    |max_range_km|total_deliveries|successful_deliveries|total_failures|success_rate_pct|avg_km_per_pct_battery|avg_battery_consumed_pct|avg_distance_km|
+--------+---------+------------+----------------+---------------------+--------------+----------------+----------------------+------------------------+---------------+
|D069    |Wing-G2  |42.3        |63              |43                   |20            |68.25           |1.984                 |10.46                   |20.15          |
|D080    |Wing-G2  |37.3        |36              |25                   |11            |69.44           |2.097                 |9.15                    |18.56          |
|D005    |Skydio-D2|61.1        |46              |32                   |14            |69.57           |1.949            

In [0]:
# View 2: Route Summary
spark.sql(f"""
    CREATE OR REPLACE VIEW {DB}.vw_route_summary AS
    SELECT
        source,
        destination,
        total_deliveries,
        avg_delivery_time_mins,
        min_delivery_time_mins,
        max_delivery_time_mins,
        avg_distance_km,
        ROUND(max_delivery_time_mins - min_delivery_time_mins, 2) AS delivery_time_variance_mins
    FROM {DB}.gold_avg_delivery_time
""")

print("vw_route_summary created")
spark.sql(f"SELECT * FROM {DB}.vw_route_summary ORDER BY total_deliveries DESC LIMIT 3").show(truncate=False)

vw_route_summary created
+---------------+-----------+----------------+----------------------+----------------------+----------------------+---------------+---------------------------+
|source         |destination|total_deliveries|avg_delivery_time_mins|min_delivery_time_mins|max_delivery_time_mins|avg_distance_km|delivery_time_variance_mins|
+---------------+-----------+----------------+----------------------+----------------------+----------------------+---------------+---------------------------+
|Warehouse-North|Zone-10    |44              |35.17                 |4.88                  |84.57                 |33.48          |79.69                      |
|Warehouse-North|Zone-06    |40              |31.66                 |5.23                  |137.28                |29.99          |132.05                     |
|Hub-West       |Zone-08    |39              |31.59                 |5.27                  |95.92                 |31.48          |90.65                      |
+--------------

In [0]:
# View 3: Failure Cause Summary
spark.sql(f"""
    CREATE OR REPLACE VIEW {DB}.vw_failure_cause_summary AS
    SELECT
        failure_cause,
        failure_count,
        pct_of_failures,
        pct_of_total_deliveries
    FROM {DB}.gold_failure_rate_by_cause
    ORDER BY failure_count DESC
""")
print("vw_failure_cause_summary created")
spark.sql(f"SELECT * FROM {DB}.vw_failure_cause_summary").show(truncate=False)


vw_failure_cause_summary created
+---------------+-------------+---------------+-----------------------+
|failure_cause  |failure_count|pct_of_failures|pct_of_total_deliveries|
+---------------+-------------+---------------+-----------------------+
|FAILED_BATTERY |335          |36.93          |6.7                    |
|FAILED_SIGNAL  |282          |31.09          |5.64                   |
|FAILED_WEATHER |203          |22.38          |4.06                   |
|UNKNOWN_FAILURE|87           |9.59           |1.74                   |
+---------------+-------------+---------------+-----------------------+



In [0]:
# View 4: Monthly Failure Trend
spark.sql(f"""
    CREATE OR REPLACE VIEW {DB}.vw_monthly_failure_trend AS
    SELECT
        year_month,
        total_deliveries,
        total_failures,
        successful_deliveries,
        failure_rate_pct,
        success_rate_pct
    FROM {DB}.gold_monthly_failure_trend
    ORDER BY year_month
""")
print("vw_monthly_failure_trend created")
spark.sql(f"SELECT * FROM {DB}.vw_monthly_failure_trend").show(truncate=False)

print("All views created successfully")

vw_monthly_failure_trend created
+----------+----------------+--------------+---------------------+----------------+----------------+
|year_month|total_deliveries|total_failures|successful_deliveries|failure_rate_pct|success_rate_pct|
+----------+----------------+--------------+---------------------+----------------+----------------+
|2025-10   |875             |153           |722                  |17.49           |82.51           |
|2025-11   |802             |143           |659                  |17.83           |82.17           |
|2025-12   |866             |172           |694                  |19.86           |80.14           |
|2026-01   |832             |144           |688                  |17.31           |82.69           |
|2026-02   |791             |137           |654                  |17.32           |82.68           |
|2026-03   |834             |158           |676                  |18.94           |81.06           |
+----------+----------------+--------------+--------------

## 2. SQL Insights

### Insight 1 — Top 5 and Bottom 5 Drones by Reliability
**Operational use:** Schedule maintenance for bottom 5 drones. Assign high-priority deliveries to top 5.

In [0]:
print("TOP 5 — Most Reliable Drones:")
spark.sql(f"""
    SELECT drone_id, model, max_range_km, total_deliveries,
           successful_deliveries, total_failures, success_rate_pct
    FROM {DB}.gold_delivery_success_rate
    ORDER BY success_rate_pct DESC
    LIMIT 5
""").show(truncate=False)

print("BOTTOM 5 — Least Reliable Drones (maintenance priority):")
spark.sql(f"""
    SELECT drone_id, model, max_range_km, total_deliveries,
           successful_deliveries, total_failures, success_rate_pct
    FROM {DB}.gold_delivery_success_rate
    ORDER BY success_rate_pct ASC
    LIMIT 5
""").show(truncate=False)

TOP 5 — Most Reliable Drones:
+--------+--------+------------+----------------+---------------------+--------------+----------------+
|drone_id|model   |max_range_km|total_deliveries|successful_deliveries|total_failures|success_rate_pct|
+--------+--------+------------+----------------+---------------------+--------------+----------------+
|D006    |DJI-X500|80.1        |49              |46                   |3             |93.88           |
|D046    |DJI-X300|59.4        |43              |40                   |3             |93.02           |
|D020    |Wing-G2 |43.2        |56              |52                   |4             |92.86           |
|D065    |DJI-X500|64.4        |56              |52                   |4             |92.86           |
|D011    |Wing-G2 |41.5        |51              |47                   |4             |92.16           |
+--------+--------+------------+----------------+---------------------+--------------+----------------+

BOTTOM 5 — Least Reliable Drones 

### Insight 2 — Performance by Drone Model
**Operational use:** Identify which drone model to invest in for future fleet expansion.

In [0]:
print("Drone Model Performance Comparison:")
spark.sql(f"""
    SELECT
        model,
        COUNT(drone_id)                          AS drone_count,
        ROUND(AVG(success_rate_pct), 2)          AS avg_success_rate_pct,
        ROUND(MIN(success_rate_pct), 2)          AS min_success_rate_pct,
        ROUND(MAX(success_rate_pct), 2)          AS max_success_rate_pct,
        SUM(total_deliveries)                    AS total_deliveries,
        SUM(total_failures)                      AS total_failures
    FROM {DB}.gold_delivery_success_rate
    GROUP BY model
    ORDER BY avg_success_rate_pct DESC
""").show(truncate=False)

Drone Model Performance Comparison:
+----------+-----------+--------------------+--------------------+--------------------+----------------+--------------+
|model     |drone_count|avg_success_rate_pct|min_success_rate_pct|max_success_rate_pct|total_deliveries|total_failures|
+----------+-----------+--------------------+--------------------+--------------------+----------------+--------------+
|DJI-X500  |21         |82.74               |71.15               |93.88               |1000            |171           |
|DJI-X300  |18         |81.9                |72.73               |93.02               |941             |170           |
|Zipline-R1|16         |81.76               |72.22               |90.91               |779             |143           |
|Wing-G2   |26         |81.42               |68.25               |92.86               |1324            |246           |
|Skydio-D2 |19         |81.29               |69.57               |91.67               |956             |177           |
+---

### Insight 3 — High-Risk Drone Identification
**Operational use:** Drones with success rate below fleet average AND above-average battery consumption are highest priority for inspection.

This is a cross-KPI query — something not possible from a single Gold table.

In [0]:
print("High-Risk Drones (below average success rate + above average battery drain):")

print("Average success rate across the fleet:",round(spark.sql(f"SELECT AVG(success_rate_pct) FROM {DB}.vw_fleet_health_dashboard").collect()[0][0],2),"%")
print("Average battery drain across the fleet:",round(spark.sql(f"SELECT AVG(avg_battery_consumed_pct) FROM {DB}.vw_fleet_health_dashboard").collect()[0][0],2),"%")

spark.sql(f"""
    WITH fleet_avg AS (
        SELECT
            AVG(success_rate_pct)       AS avg_success_rate,
            AVG(avg_battery_consumed_pct) AS avg_battery_consumed
        FROM {DB}.vw_fleet_health_dashboard
    )
    SELECT
        v.drone_id,
        v.model,
        v.success_rate_pct,
        v.avg_battery_consumed_pct,
        v.total_deliveries,
        v.total_failures
    FROM {DB}.vw_fleet_health_dashboard v
    CROSS JOIN fleet_avg fa
    WHERE v.success_rate_pct       < fa.avg_success_rate
      AND v.avg_battery_consumed_pct > fa.avg_battery_consumed
    ORDER BY v.success_rate_pct ASC
    LIMIT 10
""").show(truncate=False)

High-Risk Drones (below average success rate + above average battery drain):
Average success rate across the fleet: 81.81 %
Average battery drain across the fleet: 15.42 %
+--------+----------+----------------+------------------------+----------------+--------------+
|drone_id|model     |success_rate_pct|avg_battery_consumed_pct|total_deliveries|total_failures|
+--------+----------+----------------+------------------------+----------------+--------------+
|D005    |Skydio-D2 |69.57           |17.29                   |46              |14            |
|D064    |DJI-X500  |71.15           |16.89                   |52              |15            |
|D090    |Zipline-R1|72.22           |28.03                   |54              |15            |
|D027    |Zipline-R1|74.36           |26.81                   |39              |10            |
|D014    |DJI-X500  |74.42           |18.36                   |43              |11            |
|D009    |Zipline-R1|74.51           |20.76                 

### Insight 4 — Slowest and Busiest Routes
**Operational use:** Slow routes may need dedicated faster drone models. Busy routes may need dedicated capacity.

In [0]:
print("Top 5 Slowest Routes:")
spark.sql(f"""
    SELECT source, destination, total_deliveries,
           avg_delivery_time_mins, avg_distance_km,
           delivery_time_variance_mins
    FROM {DB}.vw_route_summary
    ORDER BY avg_delivery_time_mins DESC
    LIMIT 5
""").show(truncate=False)

print("Top 5 Busiest Routes (most deliveries):")
spark.sql(f"""
    SELECT source, destination, total_deliveries,
           avg_delivery_time_mins, avg_distance_km
    FROM {DB}.vw_route_summary
    ORDER BY total_deliveries DESC
    LIMIT 5
""").show(truncate=False)

Top 5 Slowest Routes:
+---------------+-----------+----------------+----------------------+---------------+---------------------------+
|source         |destination|total_deliveries|avg_delivery_time_mins|avg_distance_km|delivery_time_variance_mins|
+---------------+-----------+----------------+----------------------+---------------+---------------------------+
|Hub-Central    |Zone-11    |23              |41.64                 |39.48          |93.08                      |
|Warehouse-South|Zone-04    |25              |39.43                 |36.43          |75.2                       |
|Hub-West       |Zone-26    |21              |38.53                 |32.78          |97.03                      |
|Warehouse-East |Zone-21    |21              |38.44                 |36.63          |96.05                      |
|Hub-Central    |Zone-05    |29              |37.8                  |37.96          |76.35                      |
+---------------+-----------+----------------+--------------------

### Insight 5 — Failure Cause Deep Dive
**Operational use:** Battery is the leading cause - this directly informs fleet maintenance priorities.
If battery failures dominate, the fix is better pre-flight battery checks and shorter route assignments for aging drones.

In [0]:
print("Failure Cause Analysis:")
spark.sql(f"""
    SELECT
        failure_cause,
        failure_count,
        pct_of_failures,
        pct_of_total_deliveries,
        CASE
            WHEN failure_cause = 'FAILED_BATTERY'
                THEN 'Pre-flight battery checks, shorten routes for aging drones'
            WHEN failure_cause = 'FAILED_SIGNAL'
                THEN 'Improve GPS infrastructure, avoid signal-weak zones'
            WHEN failure_cause = 'FAILED_WEATHER'
                THEN 'Implement weather-based flight hold policies'
            ELSE 'Investigate sensor data for unknown failure patterns'
        END AS recommended_action
    FROM {DB}.gold_failure_rate_by_cause
    ORDER BY failure_count DESC
""").show(truncate=False)

Failure Cause Analysis:
+---------------+-------------+---------------+-----------------------+----------------------------------------------------------+
|failure_cause  |failure_count|pct_of_failures|pct_of_total_deliveries|recommended_action                                        |
+---------------+-------------+---------------+-----------------------+----------------------------------------------------------+
|FAILED_BATTERY |335          |36.93          |6.7                    |Pre-flight battery checks, shorten routes for aging drones|
|FAILED_SIGNAL  |282          |31.09          |5.64                   |Improve GPS infrastructure, avoid signal-weak zones       |
|FAILED_WEATHER |203          |22.38          |4.06                   |Implement weather-based flight hold policies              |
|UNKNOWN_FAILURE|87           |9.59           |1.74                   |Investigate sensor data for unknown failure patterns      |
+---------------+-------------+---------------+------------

### Insight 6 — Winter vs Non-Winter Failure Rate
**Operational use:** Quantifies the seasonal impact on fleet reliability.
Built on the seasonal weather patterns embedded in our simulated data.

In [0]:
print("Seasonal Failure Rate Comparison:")
spark.sql(f"""
    SELECT
        CASE
            WHEN year_month IN ('2025-12', '2026-01', '2026-02')
            THEN 'Winter (Dec-Feb)'
            ELSE 'Non-Winter (Oct-Nov, Mar)'
        END                                             AS season,
        SUM(total_deliveries)                           AS total_deliveries,
        SUM(total_failures)                             AS total_failures,
        ROUND(SUM(total_failures) /
              SUM(total_deliveries) * 100, 2)           AS failure_rate_pct,
        ROUND(SUM(successful_deliveries) /
              SUM(total_deliveries) * 100, 2)           AS success_rate_pct
    FROM {DB}.gold_monthly_failure_trend
    GROUP BY 1
    ORDER BY failure_rate_pct DESC
""").show(truncate=False)

Seasonal Failure Rate Comparison:
+-------------------------+----------------+--------------+----------------+----------------+
|season                   |total_deliveries|total_failures|failure_rate_pct|success_rate_pct|
+-------------------------+----------------+--------------+----------------+----------------+
|Winter (Dec-Feb)         |2489            |453           |18.2            |81.8            |
|Non-Winter (Oct-Nov, Mar)|2511            |454           |18.08           |81.92           |
+-------------------------+----------------+--------------+----------------+----------------+



### Insight 7 — Cross-KPI Fleet Health Ranking
**Operational use:** Ranks every drone by a combined health score using both success rate and battery efficiency.
This is the kind of query a fleet manager would run weekly.

In [0]:
print("Cross-KPI Fleet Health Ranking (Top 10):")
spark.sql(f"""
    WITH ranked AS (
        SELECT
            drone_id,
            model,
            success_rate_pct,
            avg_km_per_pct_battery,
            total_deliveries,
            ROUND(
                (success_rate_pct * 0.7) +
                (avg_km_per_pct_battery * 10 * 0.3), 2
            ) AS health_score
        FROM {DB}.vw_fleet_health_dashboard
    )
    SELECT
        RANK() OVER (ORDER BY health_score DESC) AS fleet_rank,
        drone_id,
        model,
        success_rate_pct,
        avg_km_per_pct_battery,
        total_deliveries,
        health_score
    FROM ranked
    ORDER BY fleet_rank
    LIMIT 10
""").show(truncate=False)

print("\nNote: health_score = (success_rate_pct x 0.7) + (battery_efficiency x 10 x 0.3)")
print("Weights reflect that reliability matters more than efficiency in delivery operations.")

Cross-KPI Fleet Health Ranking (Top 10):
+----------+--------+----------+----------------+----------------------+----------------+------------+
|fleet_rank|drone_id|model     |success_rate_pct|avg_km_per_pct_battery|total_deliveries|health_score|
+----------+--------+----------+----------------+----------------------+----------------+------------+
|1         |D006    |DJI-X500  |93.88           |2.009                 |49              |71.74       |
|2         |D046    |DJI-X300  |93.02           |2.057                 |43              |71.29       |
|3         |D065    |DJI-X500  |92.86           |2.06                  |56              |71.18       |
|4         |D011    |Wing-G2   |92.16           |2.208                 |51              |71.14       |
|5         |D020    |Wing-G2   |92.86           |2.026                 |56              |71.08       |
|6         |D075    |Skydio-D2 |91.67           |2.058                 |48              |70.34       |
|7         |D008    |Zipline-R1|

### Insight 8 — Destination Zone Failure Hotspots
**Operational use:** Identifies geographic zones with consistently high failure rates.
Could indicate GPS dead zones, terrain interference, or localised weather patterns.
Fleet managers can flag these zones for extra caution or route avoidance.

In [0]:
print("Top 10 Highest Risk Delivery Zones (most failures):")
spark.sql(f"""
    SELECT
        destination,
        COUNT(*)                              AS total_deliveries,
        SUM(failure_flag)                     AS total_failures,
        ROUND(SUM(failure_flag) /
              COUNT(*) * 100, 2)              AS failure_rate_pct,
        ROUND(AVG(distance_km), 2)            AS avg_distance_km,
        ROUND(AVG(delivery_duration_mins), 2) AS avg_duration_mins
    FROM {DB}.silver_deliveries
    GROUP BY destination
    HAVING COUNT(*) >= 100
    ORDER BY failure_rate_pct DESC
    LIMIT 10
""").show(truncate=False)

print("Top 5 Safest Delivery Zones (lowest failure rate):")
spark.sql(f"""
    SELECT
        destination,
        COUNT(*)                   AS total_deliveries,
        SUM(failure_flag)          AS total_failures,
        ROUND(SUM(failure_flag) /
              COUNT(*) * 100, 2)   AS failure_rate_pct
    FROM {DB}.silver_deliveries
    GROUP BY destination
    HAVING COUNT(*) >= 100
    ORDER BY failure_rate_pct ASC
    LIMIT 5
""").show(truncate=False)


Top 10 Highest Risk Delivery Zones (most failures):
+-----------+----------------+--------------+----------------+---------------+-----------------+
|destination|total_deliveries|total_failures|failure_rate_pct|avg_distance_km|avg_duration_mins|
+-----------+----------------+--------------+----------------+---------------+-----------------+
|Zone-14    |158             |39            |24.68           |29.37          |25.13            |
|Zone-18    |147             |35            |23.81           |32.78          |28.7             |
|Zone-05    |171             |40            |23.39           |31.46          |28.05            |
|Zone-25    |173             |38            |21.97           |29.0           |26.01            |
|Zone-21    |142             |31            |21.83           |28.71          |25.23            |
|Zone-22    |170             |34            |20.0            |31.65          |29.43            |
|Zone-07    |176             |35            |19.89           |30.53        

### Insight 9 — Fleet Workload Distribution
**Operational use:** Detects scheduling imbalances across the fleet.
High STDDEV relative to AVG means some drones are overworked.
A high failure rate on overworked drones is a scheduling problem - not a hardware defect.
Very different root cause, very different fix.

In [0]:
print("Fleet Workload Statistics:")
spark.sql(f"""
    SELECT
        ROUND(AVG(total_deliveries), 2)    AS avg_deliveries_per_drone,
        ROUND(STDDEV(total_deliveries), 2) AS stddev_deliveries,
        MIN(total_deliveries)              AS min_deliveries,
        MAX(total_deliveries)              AS max_deliveries,
        MAX(total_deliveries) -
        MIN(total_deliveries)              AS delivery_range
    FROM {DB}.gold_delivery_success_rate
""").show(truncate=False)

print("Workload & Reliability of drones:")
spark.sql(f"""
    WITH fleet_stats AS (
        SELECT AVG(total_deliveries) AS avg_del
        FROM {DB}.gold_delivery_success_rate
    )
    SELECT
        g.drone_id,
        g.model,
        g.total_deliveries,
        g.success_rate_pct,
        ROUND(g.total_deliveries / f.avg_del * 100, 1) AS workload_pct_wrt_avg,
        CASE
            WHEN g.total_deliveries > f.avg_del * 1.3 THEN 'Overworked'
            WHEN g.total_deliveries < f.avg_del * 0.7 THEN 'Underutilised'
            ELSE 'Normal'
        END AS workload_status
    FROM {DB}.gold_delivery_success_rate g
    CROSS JOIN fleet_stats f
    ORDER BY g.total_deliveries DESC
    LIMIT 10
""").show(truncate=False)


Fleet Workload Statistics:
+------------------------+-----------------+--------------+--------------+--------------+
|avg_deliveries_per_drone|stddev_deliveries|min_deliveries|max_deliveries|delivery_range|
+------------------------+-----------------+--------------+--------------+--------------+
|50.0                    |7.02             |33            |65            |32            |
+------------------------+-----------------+--------------+--------------+--------------+

Workload & Reliability of drones:
+--------+----------+----------------+----------------+--------------------+---------------+
|drone_id|model     |total_deliveries|success_rate_pct|workload_pct_wrt_avg|workload_status|
+--------+----------+----------------+----------------+--------------------+---------------+
|D082    |DJI-X300  |65              |86.15           |130.0               |Normal         |
|D059    |Wing-G2   |65              |78.46           |130.0               |Normal         |
|D069    |Wing-G2   |63

### Insight 10 — Failure Timing: Early vs Late Flight
**Operational use:** Determines at what point in a mission failures tend to occur.

- Failures at **<50% of journey** → takeoff or early system fault
- Failures at **50–80% of journey** → mid-flight battery/signal endurance issue
- Failures at **>80% of journey** → last-mile reliability issue

Computed as: `AVG(failed_duration / avg_successful_duration_for_same_distance_bucket) * 100`

In [0]:
print("Failure Timing -- At What Point in the Journey Do Failures Occur:")
spark.sql(f"""
    WITH success_baseline AS (
        SELECT
            CASE
                WHEN distance_km < 25              THEN 'Short'
                WHEN distance_km BETWEEN 25 AND 60 THEN 'Medium'
                ELSE                                    'Long'
            END                         AS distance_bucket,
            AVG(delivery_duration_mins) AS avg_success_duration
        FROM {DB}.silver_deliveries
        WHERE failure_flag = 0
        GROUP BY 1
    ),
    failed_deliveries AS (
        SELECT
            delivery_duration_mins,
            CASE
                WHEN distance_km < 25              THEN 'Short'
                WHEN distance_km BETWEEN 25 AND 60 THEN 'Medium'
                ELSE                                    'Long'
            END AS distance_bucket
        FROM {DB}.silver_deliveries
        WHERE failure_flag = 1
    )
    SELECT
        f.distance_bucket,
        COUNT(*)                                        AS failed_deliveries,
        ROUND(AVG(f.delivery_duration_mins), 2)         AS avg_failure_duration_mins,
        ROUND(AVG(s.avg_success_duration), 2)           AS avg_success_duration_mins,
        ROUND(AVG(f.delivery_duration_mins /
                  s.avg_success_duration) * 100, 1)     AS failure_at_what_pct_of_journey,
        ROUND(AVG(f.delivery_duration_mins /
          s.avg_success_duration) * 100, 1) AS failure_at_pct_of_journey,
        CASE
            WHEN failure_at_pct_of_journey < 50  THEN 'Early failure -- takeoff or system fault'
            WHEN failure_at_pct_of_journey <= 80 THEN 'Mid-flight -- battery or signal endurance'
            ELSE                                      'Late failure -- last-mile reliability issue'
        END AS failure_stage_interpretation
    FROM failed_deliveries f
    JOIN success_baseline s ON f.distance_bucket = s.distance_bucket
    GROUP BY f.distance_bucket
    ORDER BY f.distance_bucket
""").show(truncate=False)


Failure Timing -- At What Point in the Journey Do Failures Occur:
+---------------+-----------------+-------------------------+-------------------------+------------------------------+-------------------------+-----------------------------------------+
|distance_bucket|failed_deliveries|avg_failure_duration_mins|avg_success_duration_mins|failure_at_what_pct_of_journey|failure_at_pct_of_journey|failure_stage_interpretation             |
+---------------+-----------------+-------------------------+-------------------------+------------------------------+-------------------------+-----------------------------------------+
|Long           |67               |39.09                    |71.63                    |54.6                          |54.6                     |Mid-flight -- battery or signal endurance|
|Medium         |427              |21.98                    |38.41                    |57.2                          |57.2                     |Mid-flight -- battery or signal endurance|

## 5. Complete Insights Summary

In [0]:
print("SQL Insights Layer -- Complete Summary")
print("=" * 55)
views = ["vw_fleet_health_dashboard", "vw_route_summary", "vw_failure_cause_summary", "vw_monthly_failure_trend"]
print("\nViews:")
for v in views:
    count = spark.sql(f"SELECT COUNT(*) FROM {DB}.{v}").collect()[0][0]
    print(f"  {v:<32} {count:>6,} rows")
print("Core insights      : 7")
print("Additional insights: 3")
print("Total insights     : 10")
print("Cross-KPI queries  : 3  (Insight 3, 7, 9)")
print("Seasonal analysis  : 2  (Insight 6, 10)")
print("Geographic analysis: 1  (Insight 8)")

SQL Insights Layer -- Complete Summary

Views:
  vw_fleet_health_dashboard           100 rows
  vw_route_summary                    150 rows
  vw_failure_cause_summary              4 rows
  vw_monthly_failure_trend              6 rows
Core insights      : 7
Additional insights: 3
Total insights     : 10
Cross-KPI queries  : 3  (Insight 3, 7, 9)
Seasonal analysis  : 2  (Insight 6, 10)
Geographic analysis: 1  (Insight 8)
